In [1]:
from os import getenv
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=getenv("OPENROUTER_API_KEY"),
)

MODEL = "nvidia/nemotron-3.5-lightning:free"

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

Based on the FAQ context (Module 1: RAG), here's how to run Ollama locally:

1. **Install Ollama**  
   Visit [https://ollama.com/download](https://ollama.com/download) and install for your OS:
   - **macOS**: Download the `.pkg` and install it.
   - **Windows**: Download the `.msi` and install it.
   - **Linux**: Run in terminal: `curl -fsSL https://ollama.com/install.sh | sh`

2. **Start the model**  
   Open a terminal and run:  
   ```bash
   ollama run llama3
   ```  
   This downloads the LLaMA 3 model (~4GB), starts it locally, and opens a chat-like interface for typing questions.

3. **Test the local server**  
   Run:  
   ```bash
   curl http://localhost:11434
   ```  
   You should receive a JSON response similar to: `{"models": [...]}`.

4. **Use with Python** (optional)  
   Install the Python client: `pip install ollama`  
   Minimal example:  
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": you

In [5]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

The provided FAQ context does not contain any information about running Olama locally. The available entries cover:

- Running the course locally vs. using Codespaces (Python, `uv`, Jupyter, Docker setup)
- Why orchestration/Kestra is needed over notebooks
- Capstone project submission and running recommendations
- Using Groq instead of OpenAI in the dlt workshop
- Listing database tables via `db.table_names()`

If you're looking for guidance on any of these topics, I'd be happy to help based on the context. For questions about Olama or local LLM runtime setup, that topic isn't covered in the given FAQ snippets.


In [6]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.chat.completions.create(
    model=MODEL,
    messages=messages,
)

response.choices[0].message.content

"I'd love to help you join! But I'm not sure which course you're referring to yet. Could you share:\n\n- The course name or topic?\n- The platform (Coursera, edX, Udemy, a university, a company program, etc.)?\n- Any link or details you have?\n\nOnce I know more, I can tell you about enrollment steps, start dates, prerequisites, and how to get started. 🚀"

In [7]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}

In [9]:
response = openai_client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=[search_tool]
)

In [10]:
len(response.choices[0].message.tool_calls)

1

In [11]:
call = response.choices[0].message.tool_calls[0]

In [12]:
call

ChatCompletionMessageFunctionToolCall(id='call-33aec7f3-a3da-4fb6-b429-2e1df873d150', function=Function(arguments='{"query":"join course"}', name='search'), type='function', index=0)

In [13]:
import json

args = json.loads(call.function.arguments)
args

{'query': 'join course'}

In [14]:
call.function.name

'search'

In [15]:
results = search(**args)

In [16]:
result_json = json.dumps(results, indent=2)

In [17]:
function_call_output = {
    "role": "tool",
    "tool_call_id": call.id,
    "content": result_json,
}

In [18]:
messages.append(response.choices[0].message)

In [19]:
messages.append(function_call_output)

In [20]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call-33aec7f3-a3da-4fb6-b429-2e1df873d150', function=Function(arguments='{"query":"join course"}', name='search'), type='function', index=0)], reasoning='The user asks: "I just discovered the course. Can I join it?" We need to answer based on FAQ possibly. Use the search function to query the FAQ database for entries matching the query about joining the course. Let\'s search.', reasoning_details=[{'type': 'reasoning.text', 'text': 'The user asks: "I just discovered the course. Can I join it?" We need to answer based on FAQ possibly. Use the search function to query the FAQ database for entries matching the query about joining the course. Let\'s search.', 'format': 'unknown', 'index': 0}]),
 {'role': 'tool',
  'tool_call_id': 'call-33aec7f3

In [21]:
response = openai_client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=[search_tool]
)

In [22]:
print(response.choices[0].message.content)

Yes – you can join the course!  

According to the FAQ, “I just discovered the course. Can I still join?” the answer is:

**Yes, you can join.**  
If you want to receive a certificate, you’ll need to submit your project while the course is still accepting submissions. The course materials (videos, notebooks, homework, etc.) are available for you to start anytime, and you can work through them at your own pace. Certificate eligibility does require completing the capstone project and the required peer reviews during a live cohort, but you’re welcome to start learning right away even if you don’t plan to earn a certificate.


In [23]:
usage = response.usage
usage.prompt_tokens, usage.completion_tokens

(1111, 192)

In [24]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(usage.prompt_tokens, usage.completion_tokens)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.00028185


In [25]:
def make_call(call):
    args = json.loads(call.function.arguments)

    if call.function.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "role": "tool",
        "tool_call_id": call.id,
        "content": result_json,
    }

In [26]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [27]:
response = openai_client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=[search_tool]
)

In [28]:
message = response.choices[0].message
messages.append(message)

if message.tool_calls:
    for item in message.tool_calls:
        print('function_call:', item.function.name, item.function.arguments)
        call_output = make_call(item)
        messages.append(call_output)

if message.content:
    print('ASSISTANT:')
    print(message.content)

function_call: search {"query":"join course enrollment"}


In [29]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function.\nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call-8c823802-0ea8-488e-a4dd-b24a2f0a2f92', function=Function(arguments='{"query":"join course enrollment"}', name='search'), type='function', index=0)], reasoning='User wants to know if they can join the course. Likely need FAQ about enroll

In [30]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=[search_tool]
    )

    message = response.choices[0].message
    messages.append(message)

    if message.tool_calls:
        for item in message.tool_calls:
            print('function_call:', item.function.name, item.function.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

    if message.content:
        print('ASSISTANT:')
        print(message.content)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join course"}
iteration #2...
ASSISTANT:
Yes – you can join the course right away!  

**Key point:** If you’d like to receive a certificate, you’ll need to submit your project before the current cohort’s submission window closes. The course materials (videos, notebooks, homework, etc.) are available now, and you can start working through them whenever you like.

If you have any other questions—about the weekly workflow, homework deadlines, or how to submit your project—just let me know!


In [31]:
def agent_loop(instructions, question, model=MODEL) -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1
    last_answer = None

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=[search_tool]
        )

        message = response.choices[0].message
        messages.append(message)

        if message.tool_calls:
            for item in message.tool_calls:
                print('function_call:', item.function.name, item.function.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

        if message.content:
            print('ASSISTANT:')
            last_answer = message.content
            print(message.content)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [32]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results
and then perform more searchers.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [33]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join course enrollment"}
iteration #2...
ASSISTANT:
**Yes – you can definitely join the course!**  

The course materials (videos, slides, notebooks, and GitHub repos) are already available, so you can start learning right away. Here are a few key points to help you get started:

| Topic | What you need to know |
|-------|-----------------------|
| **When can I start?** | Anytime. The lectures and code notebooks are uploaded and don’t have a fixed start date. |
| **How do I enroll?** | No formal registration is required to access the content. If you want to keep track of your progress or receive occasional notifications, you can “register” on the course platform, but it’s not mandatory. |
| **Certificate** | A certificate is only issued to participants who complete the course **within a live cohort** (i.e., submit the capstone project and required peer reviews while the cohort is still accepting submissions). If you’re studying self‑paced

In [34]:
result

'**Yes – you can definitely join the course!**  \n\nThe course materials (videos, slides, notebooks, and GitHub repos) are already available, so you can start learning right away. Here are a few key points to help you get started:\n\n| Topic | What you need to know |\n|-------|-----------------------|\n| **When can I start?** | Anytime. The lectures and code notebooks are uploaded and don’t have a fixed start date. |\n| **How do I enroll?** | No formal registration is required to access the content. If you want to keep track of your progress or receive occasional notifications, you can “register” on the course platform, but it’s not mandatory. |\n| **Certificate** | A certificate is only issued to participants who complete the course **within a live cohort** (i.e., submit the capstone project and required peer reviews while the cohort is still accepting submissions). If you’re studying self‑paced, you can still work through all the material, but you’d need to wait for the next live coh

In [35]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results
and then perform more searchers.

The question has to be about the course or its logistics, offtopic questions
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
ASSISTANT:
I’m sorry, but “queen’s gambit” isn’t covered in the course FAQ, and it appears to be outside the scope of this course’s topics. If you have any questions about the course material, logistics, assignments, or anything else related to the course, feel free to ask!


In [36]:
from toyaikit.llm import OpenAIChatCompletionsClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIChatCompletionsRunner, DisplayingRunnerCallback

In [37]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [38]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': 'llm-zoomcamp'}
    )

In [39]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [40]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [41]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [42]:
from toyaikit.pricing import PricingConfig

# toyaikit has no price for this model (it also reads the ":free" suffix as the model name),
# so register it as $0. Without this, result.cost is None.
pricing_config = PricingConfig()
pricing_config.register_model("free", 0, 0)

runner = OpenAIChatCompletionsRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIChatCompletionsClient(model=MODEL, client=openai_client),
    pricing_config=pricing_config,
)

In [43]:
result = runner.loop(
    prompt='How do I run Olama locally?',
    callback=callback,
)

-> Response received


-> Response received


In [44]:
result.cost

CostInfo(input_cost=Decimal('0'), output_cost=Decimal('0'), total_cost=Decimal('0'))

In [45]:
result.all_messages

[{'role': 'system',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function.\nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results\nand then perform more searchers.\n\nThe question has to be about the course or its logistics, offtopic questions\nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the\nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'How do I run Olama locally?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='cal

In [46]:
result2 = runner.loop(
    prompt='How do I run a different model?',
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


Action,Command
See installed models,ollama list
Download a new model,ollama pull <model‑name>
Start a model,ollama run <model‑name>
Chat via Python,"ollama.chat(model='<model‑name>', messages=[…])"
